# M6 evaluation and human-review demo

This notebook evaluates the saved M1-M5 offline pipeline on authorised Drive media. It runs each video once, replays the five incident strategies from the saved signals, matches `active` transitions to reviewed events, and writes an auditable report. M3B and the VLM are optional; neither changes incident metrics.

## How to execute this Colab

1. Run `colab-notebooks/dataset_download.ipynb` first if you need the evaluation media. Keep media under the Drive folder configured as `CROWD_SAFETY_DATA_ROOT`.
2. Run this notebook top-to-bottom once in a fresh Colab runtime. The first code cell clones/installs the repository and mounts Drive.
3. Use the optional draft-manifest cell only when you need to discover new videos. Review its JSON and add event timestamps/labels before evaluation; do not rerun it for every evaluation.
4. Set `RUN_EVALUATION = True` in the configuration cell, then run the final execution cell. Start with one or two `SELECTED_VIDEO_IDS` while checking the setup; use `None` only for the full manifest.
5. Watch the progress lines printed by the final cell. Results are written to `CROWD_SAFETY_EVAL_ROOT` in Drive: `summary.md`, `metrics.json`, `predictions.jsonl`, `incidents.jsonl`, and per-video source runs/evidence.

The evaluation is expected to take time: every selected video is decoded, person detection/tracking and rolling VideoMAE violence inference run, then five inexpensive replays are generated. The old notebook stayed silent until all videos finished, which made a healthy run look hung. This version prints progress before and after each video.

For a quick smoke test, leave `RUN_EVALUATION = False`; the manifest, matcher, replay, model-comparison, and disabled-VLM self-checks still run without media.


In [ ]:
from __future__ import annotations

from copy import deepcopy
from dataclasses import replace
from datetime import datetime, timezone
import json, os, platform, shutil, subprocess, sys, tempfile
from pathlib import Path, PurePosixPath
from uuid import uuid4

REPO = Path('/content/realtime-crowd-safety-monitoring')
if not (REPO / 'src').is_dir():
    local_repo = Path.cwd()
    if (local_repo / 'src').is_dir():
        REPO = local_repo
    else:
        if REPO.exists():
            shutil.rmtree(REPO)
        clone = subprocess.run([
            'git', 'clone', '--depth', '1',
            'https://github.com/govardhan-06/realtime-crowd-safety-monitoring.git',
            str(REPO),
        ], capture_output=True, text=True)
        if clone.returncode != 0:
            raise RuntimeError(f'Could not clone repository: {clone.stderr.strip()}')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{REPO}[violence]'], check=True)

DRIVE_ROOT = Path('/content/drive/MyDrive/crowd_safety')
os.environ.setdefault('CROWD_SAFETY_DATA_ROOT', str(DRIVE_ROOT))
os.environ.setdefault('CROWD_SAFETY_EVAL_ROOT', os.path.join(os.environ['CROWD_SAFETY_DATA_ROOT'], 'evaluation', 'runs'))
sys.path.insert(0, str(REPO / 'src'))

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except ImportError:
    pass

DATA_ROOT = Path(os.environ['CROWD_SAFETY_DATA_ROOT']).expanduser()
MANIFEST_ROOT = DATA_ROOT / 'evaluation/manifests'
EVAL_ROOT = Path(os.environ['CROWD_SAFETY_EVAL_ROOT']).expanduser()
MANIFEST_CANDIDATES = (
    MANIFEST_ROOT / 'manifest.json',
    MANIFEST_ROOT / 'manifest-draft.json',
    REPO / 'evaluation/manifests/manifest-draft.json',
    REPO / 'evaluation/manifests/example-test.json',
)
MANIFEST_PATH = next((path for path in MANIFEST_CANDIDATES if path.is_file()), MANIFEST_CANDIDATES[-1])
CONFIG_PATH = REPO / 'configs/pipeline/dev.toml'
STRATEGIES = ('violence-only', 'crowd-only', 'naive-or', 'rule-fusion', 'temporal')
RUN_EVALUATION = False  # change to True only after the reviewed manifest and Drive media are ready
SELECTED_VIDEO_IDS = None  # e.g. ['xd-001']; keep None for every manifest entry
print({'repo': str(REPO), 'drive_root': str(DRIVE_ROOT), 'manifest': str(MANIFEST_PATH), 'evaluation_root': str(EVAL_ROOT), 'run_evaluation': RUN_EVALUATION})


## 0. Generate a draft manifest from Drive videos

Run this after `dataset_download.ipynb`. It automates file discovery and duration/model-label metadata, but leaves event timestamps for human review.

In [ ]:
# Keep the manifest local; only the referenced media lives in Drive.
DRAFT_MANIFEST_PATH = REPO / 'evaluation' / 'manifests' / 'manifest-draft.json'
DRAFT_SOURCE_DIRS = (
    (DATA_ROOT / 'poc_eval/xd_violence', 'XD-Violence', 'xd', 6),
    (DATA_ROOT / 'raw/mot20', 'MOT20', 'mot20', 3),
)

def _video_duration(path):
    command = [
        'ffprobe', '-v', 'error', '-show_entries', 'format=duration',
        '-of', 'default=noprint_wrappers=1:nokey=1', str(path)
    ]
    result = subprocess.run(command, capture_output=True, text=True)
    if result.returncode == 0 and result.stdout.strip():
        return round(float(result.stdout.strip()), 3)
    # Some Colab/Drive video files have a probe-unfriendly container but are
    # still readable by OpenCV. Use it only as a duration fallback.
    try:
        import cv2
        capture = cv2.VideoCapture(str(path))
        fps = capture.get(cv2.CAP_PROP_FPS)
        frames = capture.get(cv2.CAP_PROP_FRAME_COUNT)
        capture.release()
        if fps > 0 and frames > 0:
            return round(frames / fps, 3)
    except Exception:
        pass
    detail = result.stderr.strip().splitlines()[-1] if result.stderr.strip() else 'unknown probe error'
    raise ValueError(f'unreadable video {path}: {detail}')

def _draft_model_label(path, dataset):
    if dataset == 'XD-Violence':
        return 'normal' if '_label_A' in path.name else 'violent'
    return None

def generate_draft_manifest():
    entries = []
    unreadable = []
    for folder, dataset, prefix, limit in DRAFT_SOURCE_DIRS:
        videos = sorted(folder.glob('*.mp4'))
        if not videos:
            print(f'No MP4 files found: {folder}')
            continue
        if dataset == 'XD-Violence':
            normal = [path for path in videos if '_label_A' in path.name]
            violent = [path for path in videos if path not in normal]
            videos = normal[:limit // 2] + violent[:limit - limit // 2]
        else:
            videos = videos[:limit]
        for index, path in enumerate(videos, start=1):
            label = _draft_model_label(path, dataset)
            try:
                duration = _video_duration(path)
            except ValueError as exc:
                unreadable.append(str(exc))
                continue
            entries.append({
                'video_id': f'{prefix}-{index:03d}',
                'media': {'relative_path': str(path.relative_to(DATA_ROOT))},
                'split': 'test',
                'dataset': dataset,
                'source_id': f'{prefix}-camera-{index:03d}',
                'session_id': f'{prefix}-session-{index:03d}',
                'scenario': 'normal' if label != 'violent' else 'staged-violence',
                'duration_s': duration,
                'expected_alert': False,
                'model_label': label,
                'events': [],
                'tags': ['hard-negative'] if label == 'normal' else ['staged'],
            })
    if not entries:
        raise FileNotFoundError('No evaluation videos found; run dataset_download.ipynb first')
    manifest = {
        'schema_version': '1.0',
        'evaluation_id': 'drive-draft',
        'matching': {
            'temporal_tolerance_s': 1.0,
            'require_temporal_overlap': True,
            'actionable_state': 'active',
        },
        'entries': entries,
    }
    DRAFT_MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)
    DRAFT_MANIFEST_PATH.write_text(json.dumps(manifest, indent=2) + '\n')
    print(f'Wrote {len(entries)} draft entries to {DRAFT_MANIFEST_PATH}')
    if unreadable:
        print('Skipped unreadable videos:')
        print('\n'.join(f' - {item}' for item in unreadable))
    if not Path(MANIFEST_PATH).is_file() or Path(MANIFEST_PATH).name == 'example-test.json':
        globals()['MANIFEST_PATH'] = DRAFT_MANIFEST_PATH
    print(f'Active manifest: {MANIFEST_PATH}')
    print('Review positive clips, set expected_alert=true, and add event onset_s/end_s before using this manifest.')
    return manifest

GENERATE_DRAFT_MANIFEST = False
if GENERATE_DRAFT_MANIFEST:
    draft_manifest = generate_draft_manifest()
else:
    print('Draft generation skipped; using active manifest:', MANIFEST_PATH)

## 0.1 Select the manifest and execution mode

The manifest is the evaluation contract. It lists each video, its Drive-relative media path, dataset/split/scenario, and reviewed event intervals. The notebook uses it to resolve authorised media and compare predicted `active` incidents against human-reviewed events; it is not a model input or a replacement for the video files. For real evaluation, review the generated draft, set `expected_alert`, and add each event's `onset_s`, `end_s`, label, severity, source, and ROI where applicable.

The notebook uses this Drive root: `/content/drive/MyDrive/crowd_safety`. The preferred manifest location is `/content/drive/MyDrive/crowd_safety/evaluation/manifests/manifest.json`. If the reviewed JSON is only on your laptop, run the upload cell below, choose the local file in the picker, and it will copy it into that Drive location and make it the active manifest. Colab cannot use `/Users/...` paths directly. Set `RUN_EVALUATION = True` and optionally limit `SELECTED_VIDEO_IDS`; then continue top-to-bottom.


In [ ]:
# Optional: upload a reviewed manifest from your local machine.
# Run this cell only when MANIFEST_PATH is not already the reviewed Drive manifest.
from google.colab import files

uploaded = files.upload()  # choose manifest.json or manifest-draft.json in the local file picker
if len(uploaded) != 1:
    raise ValueError('Upload exactly one manifest JSON file')
uploaded_name, uploaded_bytes = next(iter(uploaded.items()))
if not uploaded_name.lower().endswith('.json'):
    raise ValueError('The uploaded file must be JSON')
MANIFEST_PATH = MANIFEST_ROOT / 'manifest.json'
MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)
MANIFEST_PATH.write_bytes(uploaded_bytes)
print('Uploaded active manifest:', MANIFEST_PATH)


In [ ]:
RUN_EVALUATION = True

## 1. Manifest contract and safe media resolution

The checked-in JSON schema is the interchange contract. The small validator below keeps the notebook runnable without adding a JSON-schema dependency and enforces the security/data-integrity rules that affect execution.

In [ ]:
SCENARIOS = {'normal', 'dense-crowd', 'staged-violence', 'combined-risk', 'hard-negative'}
EVENT_LABELS = {'crowd-risk', 'violence', 'combined-risk'}
SEVERITIES = {'low', 'medium', 'high', 'critical'}
TAGS = {'dense', 'lighting', 'occlusion', 'camera-motion', 'hard-negative', 'staged'}

def _path_error(value):
    path = PurePosixPath(value)
    return Path(value).is_absolute() or path.is_absolute() or '..' in path.parts

def validate_manifest(manifest):
    if not isinstance(manifest, dict): raise ValueError('manifest must be an object')
    errors = []
    if manifest.get('schema_version') != '1.0': errors.append('unsupported schema_version')
    matching = manifest.get('matching', {})
    if not isinstance(matching, dict): errors.append('matching must be an object'); matching = {}
    if not isinstance(matching.get('temporal_tolerance_s'), (int, float)) or isinstance(matching.get('temporal_tolerance_s'), bool) or matching.get('temporal_tolerance_s', -1) < 0: errors.append('invalid temporal tolerance')
    if matching.get('require_temporal_overlap') is not True: errors.append('temporal overlap must be required')
    entries = manifest.get('entries')
    if not isinstance(entries, list) or not entries: errors.append('entries must be non-empty')
    video_ids, event_ids, split_groups = set(), set(), {}
    for entry in entries or []:
        if not isinstance(entry, dict): errors.append('each entry must be an object'); continue
        video_id = entry.get('video_id')
        if not isinstance(video_id, str) or not video_id or video_id in video_ids: errors.append(f'duplicate/invalid video_id: {video_id}')
        else: video_ids.add(video_id)
        media = entry.get('media', {})
        relative_path = media.get('relative_path') if isinstance(media, dict) else None
        if not isinstance(relative_path, str) or not relative_path or _path_error(relative_path): errors.append(f'unsafe media path: {relative_path}')
        split, group = entry.get('split'), (entry.get('source_id'), entry.get('session_id'))
        if not isinstance(entry.get('dataset'), str) or not entry.get('dataset'): errors.append(f'invalid dataset: {video_id}')
        if split not in {'train', 'validation', 'test'}: errors.append(f'unsupported split: {split}')
        if not all(isinstance(value, str) and value for value in group): errors.append(f'invalid source/session: {group}')
        if all(isinstance(value, str) and value for value in group):
            previous = split_groups.setdefault(group, split)
            if previous != split: errors.append(f'split leakage for source/session: {group}')
        duration = entry.get('duration_s')
        duration_valid = isinstance(duration, (int, float)) and not isinstance(duration, bool) and duration > 0
        if not duration_valid: errors.append(f'invalid duration: {video_id}')
        if entry.get('scenario') not in SCENARIOS: errors.append(f'unsupported scenario: {entry.get("scenario")}')
        if not isinstance(entry.get('expected_alert'), bool): errors.append(f'expected_alert must be boolean: {video_id}')
        if entry.get('model_label') not in {'normal', 'violent', None}: errors.append(f'unsupported model label: {video_id}')
        tags = entry.get('tags', [])
        tags_valid = isinstance(tags, list) and all(isinstance(tag, str) for tag in tags)
        if not tags_valid or any(tag not in TAGS for tag in tags) or (tags_valid and len(tags) != len(set(tags))): errors.append(f'unsupported/duplicate tags: {video_id}')
        events = entry.get('events', [])
        if not isinstance(events, list): errors.append(f'events must be a list: {video_id}'); events = []
        for event in events:
            if not isinstance(event, dict): errors.append(f'event must be an object: {video_id}'); continue
            event_id = event.get('event_id')
            if not isinstance(event_id, str) or not event_id or event_id in event_ids: errors.append(f'duplicate/invalid event_id: {event_id}')
            else: event_ids.add(event_id)
            onset, end = event.get('onset_s'), event.get('end_s')
            if event.get('label') not in EVENT_LABELS: errors.append(f'unsupported event label: {event.get("label")}')
            if not isinstance(onset, (int, float)) or isinstance(onset, bool) or not isinstance(end, (int, float)) or isinstance(end, bool) or not duration_valid or not (0 <= onset < end <= duration): errors.append(f'invalid event interval: {event_id}')
            if not isinstance(event.get('expected_alert'), bool) or event.get('severity') not in SEVERITIES: errors.append(f'invalid event outcome: {event_id}')
        if entry.get('expected_alert') != any(isinstance(event, dict) and event.get('expected_alert') for event in events): errors.append(f'entry/event expected_alert mismatch: {video_id}')
    if errors: raise ValueError('; '.join(errors))
    return manifest

def resolve_media(entry, data_root, require_exists=True):
    relative = entry['media']['relative_path']
    if _path_error(relative): raise ValueError(f'media path must be relative and traversal-free: {relative}')
    root, candidate = Path(data_root).expanduser().resolve(), (Path(data_root) / relative).expanduser().resolve()
    try: candidate.relative_to(root)
    except ValueError as exc: raise ValueError(f'media path escapes data root: {relative}') from exc
    if require_exists and not candidate.is_file(): raise FileNotFoundError(f'manifest media is missing: {candidate}')
    return candidate

def load_manifest(path=MANIFEST_PATH, data_root=DATA_ROOT, require_files=False):
    manifest = json.loads(Path(path).read_text())
    validate_manifest(manifest)
    for entry in manifest['entries']: resolve_media(entry, data_root, require_exists=require_files)
    return manifest

In [ ]:
# Compact self-checks cover valid paths, duplicate IDs, bad intervals, traversal, labels, and split leakage.
fixture = load_manifest(REPO / 'evaluation/manifests/example-test.json', require_files=False)
assert len(fixture['entries']) == 3 and len(fixture['entries'][2]['events']) == 2

def must_reject(mutated):
    try: validate_manifest(mutated)
    except ValueError: return
    raise AssertionError('invalid manifest was accepted')

bad = deepcopy(fixture); bad['entries'][1]['video_id'] = bad['entries'][0]['video_id']; must_reject(bad)
bad = deepcopy(fixture); bad['entries'][0]['events'][0]['end_s'] = 2.0; must_reject(bad)
bad = deepcopy(fixture); bad['entries'][0]['media']['relative_path'] = '../outside.mp4'; must_reject(bad)
bad = deepcopy(fixture); bad['entries'][0]['events'][0]['label'] = 'unsupported'; must_reject(bad)
bad = deepcopy(fixture); bad['entries'][0]['events'] = [None]; must_reject(bad)
bad = deepcopy(fixture); bad['entries'][1]['source_id'] = bad['entries'][0]['source_id']; bad['entries'][1]['session_id'] = bad['entries'][0]['session_id']; bad['entries'][1]['split'] = 'train'; must_reject(bad)
print('manifest self-checks passed')

## 2. Prediction extraction, one-to-one event matching, and metrics

In [ ]:
def jsonl(path):
    path = Path(path)
    return [json.loads(line) for line in path.read_text().splitlines() if line.strip()] if path.exists() else []

def _evidence_complete(evidence_root, run_id, incident_id):
    manifest_path = Path(evidence_root) / run_id / incident_id / 'manifest.json'
    if not manifest_path.exists(): return False
    manifest = json.loads(manifest_path.read_text())
    return any(item.get('status') == 'available' for item in manifest.get('artifacts', []))

def active_alerts(video_id, strategy, replay_dir, source_run_dir, evidence_root, logical_source_id=None):
    snapshots = {}
    for row in jsonl(Path(replay_dir) / strategy / 'incidents.jsonl'): snapshots[row['incident_id']] = row
    alerts = []
    for index, transition in enumerate(jsonl(Path(replay_dir) / strategy / 'transitions.jsonl')):
        if transition.get('to_state') != 'active': continue
        incident = snapshots.get(transition['incident_id'], {})
        alerts.append({
            'prediction_id': f"{transition['incident_id']}:{transition['timestamp_s']:.6f}:{index}",
            'video_id': video_id, 'strategy': strategy, 'incident_id': transition['incident_id'],
            'source_id': logical_source_id or transition['source_id'], 'observed_source_id': transition['source_id'], 'roi': transition['region_id'],
            'predicted_onset_s': float(transition['timestamp_s']),
            'predicted_end_s': max(float(transition['timestamp_s']), float(incident.get('last_updated_at_s', transition['timestamp_s']))),
            'severity': transition.get('severity'), 'peak_risk': incident.get('peak_risk'),
            'reason_codes': incident.get('reason_codes', transition.get('reason_codes', [])),
            'evidence_complete': _evidence_complete(evidence_root, source_run_dir.name, transition['incident_id']),
        })
    return alerts

def _compatible(prediction, event):
    return ((not event.get('source_id') or prediction['source_id'] == event['source_id']) and
            (not event.get('roi') or prediction['roi'] == event['roi']))

def _overlaps(prediction, event, tolerance):
    return (prediction['predicted_onset_s'] <= event['end_s'] + tolerance and
            prediction['predicted_end_s'] >= event['onset_s'] - tolerance)

def match_events(entry, predictions, matching):
    tolerance = float(matching['temporal_tolerance_s'])
    expected = sorted((event for event in entry['events'] if event['expected_alert']), key=lambda item: item['onset_s'])
    remaining = list(sorted(predictions, key=lambda item: item['predicted_onset_s']))
    matched, matched_event_ids = [], set()
    for event in expected:
        candidates = [item for item in remaining if _compatible(item, event) and _overlaps(item, event, tolerance)]
        if not candidates: continue
        prediction = candidates[0]; remaining.remove(prediction); matched_event_ids.add(event['event_id'])
        matched.append({'event_id': event['event_id'], 'prediction_id': prediction['prediction_id'], 'delay_s': prediction['predicted_onset_s'] - event['onset_s'], 'duration_error_s': prediction['predicted_end_s'] - event['end_s']})
    duplicate_ids = {item['prediction_id'] for item in remaining if any(event['event_id'] in matched_event_ids and _compatible(item, event) and _overlaps(item, event, tolerance) for event in expected)}
    return {'matched': matched, 'duplicates': len(duplicate_ids), 'false_alerts': len(remaining) - len(duplicate_ids), 'missed': len(expected) - len(matched)}

def aggregate_strategy(entries, predictions_by_video, strategy, matching):
    camera_hours = sum(float(entry['duration_s']) for entry in entries) / 3600.0
    result = {'strategy': strategy, 'expected_events': 0, 'alerts': 0, 'true_positives': 0, 'false_alerts': 0, 'duplicates': 0, 'missed_events': 0, 'detection_delays_s': [], 'duration_errors_s': [], 'evidence_complete_alerts': 0, 'entry_results': []}
    for entry in entries:
        predictions = predictions_by_video.get(entry['video_id'], [])
        outcome = match_events(entry, predictions, matching)
        result['expected_events'] += sum(event['expected_alert'] for event in entry['events'])
        result['alerts'] += len(predictions); result['true_positives'] += len(outcome['matched'])
        result['false_alerts'] += outcome['false_alerts']; result['duplicates'] += outcome['duplicates']; result['missed_events'] += outcome['missed']
        result['detection_delays_s'] += [item['delay_s'] for item in outcome['matched']]
        result['duration_errors_s'] += [item['duration_error_s'] for item in outcome['matched']]
        result['evidence_complete_alerts'] += sum(item['evidence_complete'] for item in predictions)
        result['entry_results'].append({'video_id': entry['video_id'], 'dataset': entry['dataset'], 'source_id': entry['source_id'], 'scenario': entry['scenario'], 'tags': entry['tags'], **outcome, 'alert_count': len(predictions)})
    tp, fp, fn = result['true_positives'], result['false_alerts'], result['missed_events']
    result.update({'precision': tp / (tp + fp) if tp + fp else 0.0, 'recall': tp / (tp + fn) if tp + fn else 0.0,
        'f1': 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else 0.0,
        'camera_hours': camera_hours, 'false_alerts_per_camera_hour': fp / camera_hours if camera_hours else 0.0,
        'duplicate_alerts_per_true_event': result['duplicates'] / result['expected_events'] if result['expected_events'] else 0.0,
        'alert_rate_per_camera_hour': result['alerts'] / camera_hours if camera_hours else 0.0,
        'detection_delay_s_mean': sum(result['detection_delays_s']) / len(result['detection_delays_s']) if result['detection_delays_s'] else None,
        'duration_error_s_mean': sum(result['duration_errors_s']) / len(result['duration_errors_s']) if result['duration_errors_s'] else None,
        'evidence_completeness': result['evidence_complete_alerts'] / result['alerts'] if result['alerts'] else None})
    return result

In [ ]:
# Hand-authored matcher self-check: match, miss, false alert, duplicate, ROI mismatch, and delay.
test_entry = {'video_id': 'synthetic', 'source_id': 'camera', 'session_id': 's', 'scenario': 'combined-risk', 'duration_s': 10, 'expected_alert': True, 'events': [{'event_id': 'e1', 'label': 'combined-risk', 'onset_s': 3, 'end_s': 5, 'expected_alert': True, 'severity': 'high', 'source_id': 'camera', 'roi': 'main'}, {'event_id': 'e2', 'label': 'violence', 'onset_s': 7, 'end_s': 8, 'expected_alert': True, 'severity': 'medium', 'source_id': 'camera', 'roi': 'side'}], 'tags': [], '_matching': {'temporal_tolerance_s': 1, 'require_temporal_overlap': True}}
prediction = {'prediction_id': 'p1', 'source_id': 'camera', 'roi': 'main', 'predicted_onset_s': 4, 'predicted_end_s': 6, 'evidence_complete': True}
duplicate = {**prediction, 'prediction_id': 'p2', 'predicted_onset_s': 4.5}
false_alert = {**prediction, 'prediction_id': 'p3', 'roi': 'other', 'predicted_onset_s': 9, 'predicted_end_s': 9.5}
outcome = match_events(test_entry, [prediction, duplicate, false_alert], test_entry['_matching'])
assert len(outcome['matched']) == 1 and outcome['duplicates'] == 1 and outcome['false_alerts'] == 1 and outcome['missed'] == 1
assert outcome['matched'][0]['delay_s'] == 1
print('matcher self-checks passed')

## 3. Run once, replay five strategies, and write the report

In [ ]:
from crowd_safety.artifacts import config_hash, resolved_config
from crowd_safety.config import load_config
from crowd_safety.persistence import MemoryPersistence, import_run
from crowd_safety.replay import replay_run
from crowd_safety.runner import process_video

def write_jsonl(path, rows):
    Path(path).write_text(''.join(json.dumps(row, sort_keys=True) + '\n' for row in rows))

def stage_health_summary(source_runs):
    return {str(run['video_id']): json.loads((Path(run['source_run']) / 'metrics.json').read_text()).get('stage_health', {}) for run in source_runs if run['status'] == 'success'}

def model_predictions_m3a(source_runs, threshold):
    rows = []
    for run in source_runs:
        if run['status'] != 'success': continue
        evidence = jsonl(Path(run['source_run']) / 'violence.jsonl')
        valid = [item['evidence'] for item in evidence if item['evidence'].get('status') == 'available' and item['evidence'].get('score') is not None]
        meta = json.loads((Path(run['source_run']) / 'metadata.json').read_text())
        provenance = meta.get('provenance') or {}
        scores = [float(item['score']) for item in valid]
        rows.append({'model': 'M3A', 'video_id': run['video_id'], 'checkpoint': provenance.get('violence_model', 'unknown'), 'split': 'test', 'score': max(scores) if scores else None, 'threshold': threshold, 'label': 'violent' if scores and max(scores) >= threshold else ('normal' if scores else None), 'latency_ms': sum(float(item.get('latency_ms') or 0) for item in valid) / len(valid) if valid else None, 'status': 'available' if scores else 'unavailable'})
    return rows

def score_model_predictions(rows, entries):
    targets = {entry['video_id']: entry.get('model_label') for entry in entries}
    usable = [row for row in rows if row.get('status') == 'available' and row.get('score') is not None and targets.get(row['video_id']) in {'normal', 'violent'}]
    threshold = float(rows[0].get('threshold', 0.5)) if rows else 0.5
    tp = sum(row['score'] >= threshold and targets[row['video_id']] == 'violent' for row in usable)
    tn = sum(row['score'] < threshold and targets[row['video_id']] == 'normal' for row in usable)
    fp = sum(row['score'] >= threshold and targets[row['video_id']] == 'normal' for row in usable)
    fn = sum(row['score'] < threshold and targets[row['video_id']] == 'violent' for row in usable)
    pr_curve, roc_curve = [], []
    for curve_threshold in [index / 10 for index in range(11)]:
        curve_tp = sum(row['score'] >= curve_threshold and targets[row['video_id']] == 'violent' for row in usable)
        curve_tn = sum(row['score'] < curve_threshold and targets[row['video_id']] == 'normal' for row in usable)
        curve_fp = sum(row['score'] >= curve_threshold and targets[row['video_id']] == 'normal' for row in usable)
        curve_fn = sum(row['score'] < curve_threshold and targets[row['video_id']] == 'violent' for row in usable)
        pr_curve.append({'threshold': curve_threshold, 'precision': curve_tp / (curve_tp + curve_fp) if curve_tp + curve_fp else 0.0, 'recall': curve_tp / (curve_tp + curve_fn) if curve_tp + curve_fn else 0.0})
        roc_curve.append({'threshold': curve_threshold, 'tpr': curve_tp / (curve_tp + curve_fn) if curve_tp + curve_fn else 0.0, 'fpr': curve_fp / (curve_fp + curve_tn) if curve_fp + curve_tn else 0.0})
    return {'samples': len(usable), 'tp': tp, 'tn': tn, 'fp': fp, 'fn': fn, 'precision': tp / (tp + fp) if tp + fp else 0.0, 'recall': tp / (tp + fn) if tp + fn else 0.0, 'f1': 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else 0.0, 'confusion_matrix': [[tn, fp], [fn, tp]], 'pr_curve': pr_curve, 'roc_curve': roc_curve, 'status': 'available' if usable else 'pending-compatible-scores'}

def vlm_disabled_record(incident_id):
    return {'schema_version': '1.0', 'incident_id': incident_id, 'status': 'disabled', 'provider': 'disabled', 'model': '', 'text': '', 'grounded': None, 'contradicts_reasons': None, 'unsupported_details': [], 'latency_ms': None, 'reviewer_note': 'VLM disabled; deterministic incident evidence remains authoritative.'}

def markdown_table(rows, columns):
    if not rows: return '_none_'
    header = '| ' + ' | '.join(columns) + ' |\n| ' + ' | '.join('---' for _ in columns) + ' |'
    return header + '\n' + '\n'.join('| ' + ' | '.join(str(row.get(column, '')) for column in columns) + ' |' for row in rows)

def failure_slice_rows(strategy_metrics, failures):
    rows = []
    for strategy, metric in strategy_metrics.items():
        for item in metric['entry_results']:
            category = 'missed_event' if item['missed'] else ('duplicate_alert' if item['duplicates'] else ('false_alert' if item['false_alerts'] else 'no_failure'))
            rows.append({'strategy': strategy, 'video_id': item['video_id'], 'dataset': item['dataset'], 'source_id': item['source_id'], 'scenario': item['scenario'], 'tags': ','.join(item['tags']) or '-', 'category': category, 'alerts': item['alert_count'], 'missed': item['missed'], 'false_alerts': item['false_alerts'], 'duplicates': item['duplicates'], 'error': ''})
    rows.extend({'strategy': '-', 'video_id': failure['video_id'], 'dataset': '-', 'source_id': '-', 'scenario': failure.get('scenario', '-'), 'tags': '-', 'category': 'run_failure', 'alerts': 0, 'missed': '-', 'false_alerts': '-', 'duplicates': '-', 'error': failure.get('error', '')} for failure in failures)
    return rows

def render_report(eval_dir, manifest, strategy_metrics, model_rows, model_status, vlm_rows, failures, failure_slices, latency_rows):
    template = (REPO / 'evaluation/templates/summary.md').read_text()
    strategy_rows = [{key: metric.get(key) for key in ('strategy', 'expected_events', 'alerts', 'precision', 'recall', 'f1', 'false_alerts_per_camera_hour', 'duplicate_alerts_per_true_event', 'detection_delay_s_mean', 'evidence_completeness')} for metric in strategy_metrics.values()]
    model_metrics = [{'model': name, **score_model_predictions(rows, manifest['entries'])} for name, rows in model_rows.items()]
    if model_status.startswith('pending'): model_metrics.append({'model': 'M3B', 'status': model_status})
    failure_text = markdown_table(failure_slices, ['strategy', 'video_id', 'dataset', 'source_id', 'scenario', 'tags', 'category', 'alerts', 'missed', 'false_alerts', 'duplicates', 'error'])
    strategy_text = markdown_table(strategy_rows, ['strategy', 'expected_events', 'alerts', 'precision', 'recall', 'f1', 'false_alerts_per_camera_hour', 'duplicate_alerts_per_true_event', 'detection_delay_s_mean', 'evidence_completeness'])
    model_text = markdown_table(model_metrics, ['model', 'status', 'samples', 'precision', 'recall', 'f1', 'confusion_matrix'])
    vlm_text = f"{len(vlm_rows)} review records; disabled/unavailable VLM status is excluded from incident metrics."
    latency_text = markdown_table(latency_rows, ['video_id', 'total_seconds', 'effective_fps', 'violence_seconds', 'fusion_seconds'])
    limitations = 'M3B is pending unless CROWD_SAFETY_M3B_PREDICTIONS points to a compatible held-out export. Results are limited by the selected annotations and remain offline engineering evidence.'
    values = {'evaluation_id': manifest['evaluation_id'], 'generated_at': datetime.now(timezone.utc).isoformat(), 'entry_count': len(manifest['entries']), 'camera_hours': round(sum(float(entry['duration_s']) for entry in manifest['entries']) / 3600, 4), 'vlm_status': 'disabled', 'm6b_status': model_status, 'strategy_table': strategy_text, 'model_table': model_text, 'failure_slices': failure_text, 'latency_summary': latency_text, 'vlm_review': vlm_text, 'limitations': limitations}
    for key, value in values.items(): template = template.replace('{{' + key + '}}', str(value))
    (Path(eval_dir) / 'summary.md').write_text(template)
    return model_metrics

def materialize_evaluation(manifest, config_path=CONFIG_PATH, data_root=DATA_ROOT, evaluation_root=EVAL_ROOT, selected_ids=None):
    selected = [entry for entry in manifest['entries'] if selected_ids is None or entry['video_id'] in selected_ids]
    if not selected: raise ValueError('selected_ids did not select any manifest entries')
    eval_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid4().hex[:8]
    eval_dir = Path(evaluation_root) / eval_id; source_root = eval_dir / 'source_runs'; evidence_root = eval_dir / 'evidence'
    source_root.mkdir(parents=True, exist_ok=False); evidence_root.mkdir(parents=True, exist_ok=True)
    base = load_config(config_path)
    config = replace(base, output_directory=source_root, m5=replace(base.m5, evidence_root=evidence_root))
    (eval_dir / 'config.json').write_text(json.dumps({'manifest_matching': manifest['matching'], 'pipeline': config.as_dict()}, indent=2, sort_keys=True, default=str) + '\n')
    (eval_dir / 'manifest.json').write_text(json.dumps(manifest, indent=2, sort_keys=True) + '\n')
    revision = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=REPO, capture_output=True, text=True).stdout.strip() or None
    (eval_dir / 'environment.json').write_text(json.dumps({'python': sys.version, 'platform': platform.platform(), 'git_revision': revision, 'data_root': str(Path(data_root).resolve())}, indent=2, sort_keys=True) + '\n')
    runs, failures, predictions_by_strategy = [], [], {strategy: {} for strategy in STRATEGIES}
    for index, entry in enumerate(selected, start=1):
        print(f'[{index}/{len(selected)}] processing {entry["video_id"]}; detector + rolling violence inference may take a while', flush=True)
        try:
            media = resolve_media(entry, data_root, require_exists=True)
            result = process_video(config, input_override=media)
            replay_run(result.run_directory, config, STRATEGIES)
            run = {'video_id': entry['video_id'], 'status': 'success', 'source_run': str(result.run_directory), 'source_run_id': result.run_id}
            runs.append(run)
            print(f'[{index}/{len(selected)}] complete {entry["video_id"]}: {result.run_directory}', flush=True)
            for strategy in STRATEGIES: predictions_by_strategy[strategy][entry['video_id']] = active_alerts(entry['video_id'], strategy, result.run_directory / 'replay', result.run_directory, evidence_root, entry['source_id'])
        except Exception as exc:
            failure = {'video_id': entry['video_id'], 'scenario': entry['scenario'], 'status': 'failed', 'error_type': type(exc).__name__, 'error': str(exc)}
            runs.append(failure); failures.append(failure)
            print(f'[{index}/{len(selected)}] failed {entry["video_id"]}: {failure["error"]}', flush=True)
    for strategy in STRATEGIES:
        for video_id, rows in predictions_by_strategy[strategy].items():
            for row in rows: row['source_run_id'] = next(run['source_run_id'] for run in runs if run['video_id'] == video_id)
    successful_entries = [entry for entry in selected if any(run['video_id'] == entry['video_id'] and run['status'] == 'success' for run in runs)]
    strategy_metrics = {strategy: aggregate_strategy(successful_entries, predictions_by_strategy[strategy], strategy, manifest['matching']) for strategy in STRATEGIES}
    all_predictions = [row for strategy in STRATEGIES for rows in predictions_by_strategy[strategy].values() for row in rows]
    all_incidents = [{'strategy': row['strategy'], 'video_id': row['video_id'], **row} for row in all_predictions]
    write_jsonl(eval_dir / 'predictions.jsonl', all_predictions); write_jsonl(eval_dir / 'incidents.jsonl', all_incidents)
    failure_slices = failure_slice_rows(strategy_metrics, failures)
    latency_rows = [{'video_id': run['video_id'], **{key: json.loads((Path(run['source_run']) / 'metrics.json').read_text()).get(key) for key in ('total_seconds', 'effective_fps', 'violence_seconds', 'fusion_seconds')}} for run in runs if run['status'] == 'success']
    metrics_payload = {'strategies': {name: {key: value for key, value in metric.items() if key != 'entry_results'} for name, metric in strategy_metrics.items()}, 'stage_health': stage_health_summary(runs), 'latency': latency_rows, 'failures': failures, 'failure_slices': failure_slices}
    (eval_dir / 'metrics.json').write_text(json.dumps(metrics_payload, indent=2, sort_keys=True, default=str) + '\n')
    m3a_rows = model_predictions_m3a(runs, config.violence.threshold)
    model_rows = {'M3A': m3a_rows}; m3b_path = os.environ.get('CROWD_SAFETY_M3B_PREDICTIONS'); model_status = 'pending-m3b'
    if m3b_path and Path(m3b_path).is_file():
        try: payload = json.loads(Path(m3b_path).read_text())
        except (OSError, ValueError): payload = {}
        if not isinstance(payload, dict): payload = {}
        target_ids = {entry['video_id'] for entry in selected if entry.get('model_label') in {'normal', 'violent'}}
        prediction_ids = [row.get('video_id') for row in payload.get('predictions', []) if isinstance(row, dict)]
        compatible = (payload.get('schema_version') == '1.0' and payload.get('model') == 'M3B' and payload.get('split') == 'test' and isinstance(payload.get('threshold'), (int, float)) and isinstance(payload.get('checkpoint'), str) and bool(payload.get('predictions')) and set(prediction_ids) == target_ids and len(prediction_ids) == len(set(prediction_ids)) and all(isinstance(row, dict) and row.get('video_id') in target_ids and row.get('status') in {'available', 'degraded', 'unavailable'} and (row.get('score') is None or isinstance(row.get('score'), (int, float))) for row in payload['predictions']))
        if compatible:
            model_rows['M3B'] = [{**row, 'model': 'M3B', 'checkpoint': payload['checkpoint'], 'split': payload['split'], 'threshold': payload['threshold']} for row in payload['predictions']]
            model_status = 'available'
        else: model_status = 'pending-incompatible-m3b'
    write_jsonl(eval_dir / 'model_predictions.jsonl', [row for rows in model_rows.values() for row in rows])
    vlm_rows = [vlm_disabled_record(row['incident_id']) for row in all_predictions]
    write_jsonl(eval_dir / 'vlm_reviews.jsonl', vlm_rows)
    model_metrics = render_report(eval_dir, manifest, strategy_metrics, model_rows, model_status, vlm_rows, failures, failure_slices, latency_rows)
    metrics_payload['models'] = {row['model']: row for row in model_metrics}
    (eval_dir / 'metrics.json').write_text(json.dumps(metrics_payload, indent=2, sort_keys=True, default=str) + '\n')
    return eval_dir, {'runs': runs, 'strategies': strategy_metrics, 'model_metrics': model_metrics, 'vlm_reviews': vlm_rows, 'failures': failures}

In [ ]:
# Prove the five-strategy replay layout without requiring media or a model.
from crowd_safety.config import load_config
with tempfile.TemporaryDirectory() as tmp:
    root = Path(tmp); source = root / 'source.mp4'; config_path = root / 'pipeline.toml'; artifact_dir = root / 'artifacts'
    config_path.write_text(f'[input]\npath = "{source}"\n[output]\ndirectory = "{artifact_dir}"\n[processing]\nresize = [16, 12]\n')
    config = load_config(config_path); run = root / 'run'; run.mkdir()
    values = resolved_config(config, config.input_path); (run / 'config.json').write_text(json.dumps({'config_hash': config_hash(values), 'config': values}))
    (run / 'features.jsonl').write_text(json.dumps({'features': [{'source_id': 'camera', 'roi_name': 'zone', 'timestamp_s': 1.0, 'status': 'available', 'occupancy': 2}]}) + '\n')
    (run / 'violence.jsonl').write_text(json.dumps({'evidence': {'source_id': 'camera', 'region_id': None, 'clip_start_s': 0.0, 'clip_end_s': 1.0, 'score': 0.9, 'model': 'fixture', 'revision': 'fixture', 'label_mapping': [['safe', 0], ['unsafe', 1]], 'status': 'available'}}) + '\n')
    replay_run(run, config, STRATEGIES)
    assert {path.name for path in (run / 'replay').iterdir()} == set(STRATEGIES) | {'metadata.json'}
print('synthetic replay self-check passed')

In [ ]:
model_fixture_entries = [{'video_id': 'normal', 'model_label': 'normal'}, {'video_id': 'violent', 'model_label': 'violent'}]
model_fixture_rows = [{'video_id': 'normal', 'score': 0.1, 'threshold': 0.5, 'status': 'available'}, {'video_id': 'violent', 'score': 0.9, 'threshold': 0.5, 'status': 'available'}]
model_fixture_metrics = score_model_predictions(model_fixture_rows, model_fixture_entries)
assert model_fixture_metrics['f1'] == 1.0 and model_fixture_metrics['pr_curve'] and model_fixture_metrics['roc_curve']
assert score_model_predictions([], model_fixture_entries)['status'] == 'pending-compatible-scores'
disabled_review = vlm_disabled_record('fixture-incident')
assert disabled_review['status'] == 'disabled' and disabled_review['grounded'] is None and disabled_review['contradicts_reasons'] is None
print('model-comparison and disabled-VLM self-checks passed')

## 4. Execute the evaluation

Set `RUN_EVALUATION = True` in the bootstrap/configuration cell before running the code cell below. The pipeline is intentionally offline and human-reviewed: it creates evidence and a report but does not dispatch alerts or contact emergency services.


In [ ]:
def start_ephemeral_review(source_run, config):
    from fastapi.testclient import TestClient
    from crowd_safety.api import create_app
    store = MemoryPersistence(); imported = import_run(source_run, store, config.m5.evidence_root)
    client = TestClient(create_app(store, config.m5.evidence_root))
    health = client.get('/health').json()
    records = store.list_incidents()
    if not records: return {'imported': imported, 'health': health, 'status': 'no_incident_to_review'}
    incident_id = records[0]['incident']['incident_id']
    response = client.post(f'/incidents/{incident_id}/acknowledge', json={'actor': 'colab-reviewer', 'timestamp': datetime.now(timezone.utc).isoformat(), 'note': 'M6 offline demo review'})
    response.raise_for_status()
    return {'imported': imported, 'health': health, 'status': 'acknowledged', 'action': response.json()}

if RUN_EVALUATION:
    manifest = load_manifest(MANIFEST_PATH, DATA_ROOT, require_files=True)
    evaluation_dir, result = materialize_evaluation(manifest, selected_ids=SELECTED_VIDEO_IDS)
    successful = next((run for run in result['runs'] if run['status'] == 'success'), None)
    if successful:
        base_config = load_config(CONFIG_PATH); review_config = replace(base_config, m5=replace(base_config.m5, evidence_root=Path(evaluation_dir) / 'evidence'))
        review = start_ephemeral_review(successful['source_run'], review_config)
        (evaluation_dir / 'demo-review.json').write_text(json.dumps(review, indent=2, sort_keys=True, default=str) + '\n')
    print('evaluation directory:', evaluation_dir)
    print('completed:', len([run for run in result['runs'] if run['status'] == 'success']), '/', len(result['runs']))
    print('failures:', result['failures'])
else:
    print('self-check mode; set RUN_EVALUATION = True for authorised Drive execution')